# 02 — Logs structurés avec structlog

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer l'intérêt des logs structurés (JSON) par rapport aux logs textuels ;
- configurer `structlog` avec des processeurs ;
- binder du contexte aux logs (request_id, user, etc.) ;
- intégrer `structlog` avec le `logging` standard ;
- utiliser les logs structurés avec des outils d'observabilité.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le module `logging` standard (handlers, formatters, niveaux) ;
- les décorateurs et context managers ;
- le format JSON ;
- les bases du déploiement (logs en production).

## Plan

1. Pourquoi des logs structurés ?
2. `structlog` — installation et premier log
3. Processeurs (processors)
4. Context binding
5. Intégration avec `logging` standard
6. Configuration pour dev vs production
7. Bonnes pratiques
8. Synthèse
9. Exercices
10. Ressources

---

## 1. Pourquoi des logs structurés ?

### Log textuel classique

```
2026-04-14 10:23:45 ERROR Paiement échoué pour user_42 montant=150.00 raison=carte_expirée
```

### Log structuré JSON

```json
{"timestamp": "2026-04-14T10:23:45Z", "level": "error", "event": "paiement_echoue", "user_id": 42, "montant": 150.00, "raison": "carte_expirée"}
```

| Critère | Log texte | Log structuré |
|---|---|---|
| Parsing | Regex fragiles | Natif JSON |
| Filtrage | `grep` / `awk` | `jq`, Elasticsearch, Loki |
| Contexte | Incrusté dans la chaîne | Champs séparés |
| Dashboard | Manuel | Automatique (Grafana, Kibana) |
| Alertes | Regex | Requêtes structurées |

---

## 2. `structlog` — installation et premier log

> **Installation :** `pip install structlog`

In [ ]:
try:
    import structlog
    print(f"structlog version : {structlog.__version__}")
except ImportError:
    print("structlog non installé — pip install structlog")

### Premier log

In [ ]:
try:
    import structlog

    log = structlog.get_logger()
    log.info("application_demarree", version="1.0.0", environnement="dev")

except ImportError:
    print("structlog non installé")

Par défaut, `structlog` affiche en mode **développement** (coloré, lisible). En production, on configure la sortie JSON.

### Niveaux de log

In [ ]:
try:
    import structlog
    log = structlog.get_logger()

    log.debug("message_debug", detail="verbose")
    log.info("message_info", action="traitement")
    log.warning("message_warning", tentatives=3)
    log.error("message_error", code=500, raison="timeout")
    log.critical("message_critical", service="paiement")

except ImportError:
    print("structlog non installé")

---

## 3. Processeurs (processors)

Les **processeurs** sont des fonctions chaînées qui transforment l'événement de log avant sa sortie. C'est le coeur de `structlog`.

In [ ]:
try:
    import structlog

    structlog.configure(
        processors=[
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.StackInfoRenderer(),
            structlog.processors.format_exc_info,
            structlog.dev.ConsoleRenderer(),  # dev : coloré
        ],
    )

    log = structlog.get_logger()
    log.info("config_chargee", nb_processeurs=5)

except ImportError:
    print("structlog non installé")

### Processeurs courants

| Processeur | Rôle |
|---|---|
| `add_log_level` | Ajoute le champ `level` |
| `TimeStamper(fmt="iso")` | Ajoute le timestamp ISO 8601 |
| `StackInfoRenderer()` | Rend les stack traces |
| `format_exc_info` | Formate les exceptions |
| `JSONRenderer()` | Sortie JSON (production) |
| `ConsoleRenderer()` | Sortie colorée (dev) |

### Écrire un processeur personnalisé

In [ ]:
try:
    import structlog

    def ajouter_hostname(logger, method_name, event_dict):
        import socket
        event_dict["hostname"] = socket.gethostname()
        return event_dict

    def masquer_secrets(logger, method_name, event_dict):
        for cle in list(event_dict.keys()):
            if "password" in cle.lower() or "secret" in cle.lower():
                event_dict[cle] = "***MASKED***"
        return event_dict

    structlog.configure(
        processors=[
            ajouter_hostname,
            masquer_secrets,
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.dev.ConsoleRenderer(),
        ],
    )

    log = structlog.get_logger()
    log.info("connexion", user="alice", password="secret123")

except ImportError:
    print("structlog non installé")

---

## 4. Context binding

Le **binding** permet d'attacher du contexte persistant à un logger. Toutes les entrées suivantes incluront automatiquement ce contexte.

In [ ]:
try:
    import structlog

    structlog.configure(
        processors=[
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.dev.ConsoleRenderer(),
        ],
    )

    log = structlog.get_logger()

    # Binder du contexte
    log = log.bind(request_id="req-abc-123", user_id=42)

    log.info("debut_traitement")
    log.info("etape_1", detail="parsing")
    log.info("etape_2", detail="validation")
    log.info("fin_traitement", duree_ms=145)

except ImportError:
    print("structlog non installé")

### `unbind` et `new`

In [ ]:
try:
    import structlog

    log = structlog.get_logger()
    log = log.bind(session="sess-001")

    # Retirer une clé
    log = log.unbind("session")
    log.info("sans_session")

    # Créer un logger frais (sans contexte)
    log2 = log.new()
    log2.info("logger_frais")

except ImportError:
    print("structlog non installé")

### Context local (par thread/coroutine)

In [ ]:
try:
    import structlog

    structlog.configure(
        processors=[
            structlog.contextvars.merge_contextvars,
            structlog.stdlib.add_log_level,
            structlog.dev.ConsoleRenderer(),
        ],
    )

    # Contexte global pour la requête courante
    structlog.contextvars.clear_contextvars()
    structlog.contextvars.bind_contextvars(request_id="req-xyz-789")

    log = structlog.get_logger()
    log.info("traitement", etape="debut")

    # Dans une autre fonction, le contexte est hérité
    def sous_traitement():
        log2 = structlog.get_logger()
        log2.info("sous_traitement", detail="ok")

    sous_traitement()
    structlog.contextvars.clear_contextvars()

except ImportError:
    print("structlog non installé")

---

## 5. Intégration avec `logging` standard

`structlog` peut s'intégrer avec le `logging` standard de Python, ce qui permet d'utiliser `structlog` dans votre code tout en bénéficiant des handlers et formatters existants.

In [ ]:
try:
    import structlog
    import logging

    # Configuration pour que structlog utilise logging en backend
    structlog.configure(
        processors=[
            structlog.stdlib.filter_by_level,
            structlog.stdlib.add_logger_name,
            structlog.stdlib.add_log_level,
            structlog.stdlib.PositionalArgumentsFormatter(),
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.StackInfoRenderer(),
            structlog.processors.format_exc_info,
            structlog.stdlib.ProcessorFormatter.wrap_for_formatter,
        ],
        logger_factory=structlog.stdlib.LoggerFactory(),
        wrapper_class=structlog.stdlib.BoundLogger,
    )

    # Configurer le logging standard
    formatter = structlog.stdlib.ProcessorFormatter(
        processor=structlog.dev.ConsoleRenderer(),
    )

    handler = logging.StreamHandler()
    handler.setFormatter(formatter)

    root_logger = logging.getLogger()
    root_logger.addHandler(handler)
    root_logger.setLevel(logging.INFO)

    # Utiliser structlog
    log = structlog.get_logger("mon_app")
    log.info("integration_logging", status="ok")

except ImportError:
    print("structlog non installé")

---

## 6. Configuration pour dev vs production

In [ ]:
try:
    import structlog
    import os

    def configurer_logs():
        env = os.environ.get("APP_ENV", "dev")

        shared_processors = [
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.StackInfoRenderer(),
            structlog.processors.format_exc_info,
        ]

        if env == "production":
            # Production : JSON pur (pour Elasticsearch, Loki, etc.)
            shared_processors.append(structlog.processors.JSONRenderer())
        else:
            # Dev : sortie colorée lisible
            shared_processors.append(structlog.dev.ConsoleRenderer())

        structlog.configure(processors=shared_processors)

    configurer_logs()
    log = structlog.get_logger()
    log.info("config_active", env=os.environ.get("APP_ENV", "dev"))

except ImportError:
    print("structlog non installé")

### Sortie JSON en production

In [ ]:
try:
    import structlog

    structlog.configure(
        processors=[
            structlog.stdlib.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.JSONRenderer(),
        ],
    )

    log = structlog.get_logger()
    log.info("requete_traitee", method="POST", path="/api/users", status=201, duree_ms=45)

except ImportError:
    print("structlog non installé")

Sortie (copiable directement dans Elasticsearch, Loki, CloudWatch) :
```json
{"event": "requete_traitee", "level": "info", "timestamp": "2026-04-14T10:30:00Z", "method": "POST", "path": "/api/users", "status": 201, "duree_ms": 45}
```

---

## 7. Bonnes pratiques

1. **Nommez vos événements** avec des snake_case (pas de phrases) : `user_created`, pas `"Utilisateur créé avec succès"`.
2. **Structurez les données** : `log.info("paiement", montant=42.0)` pas `log.info(f"Paiement de 42.0 euros")`.
3. **Bindez le contexte tôt** : request_id, user_id, session dès l'entrée de la requête.
4. **Masquez les données sensibles** : mot de passe, carte bancaire, token.
5. **JSON en production, console en dev** — jamais l'inverse.
6. **Un seul log par événement** : pas de multi-lignes inutiles.
7. **Incluez les erreurs** avec `exc_info=True` ou `log.exception()`.

### Nommer les événements

In [ ]:
try:
    import structlog
    log = structlog.get_logger()

    # BON — événement nommé, données structurées
    log.info("commande_creee", commande_id="cmd-123", montant=42.0, articles=3)

    # MAUVAIS — phrase, données incrustées
    # log.info(f"La commande cmd-123 a été créée pour 42.0 euros avec 3 articles")

except ImportError:
    print("structlog non installé")

### Logger les exceptions

In [ ]:
try:
    import structlog
    log = structlog.get_logger()

    try:
        result = 1 / 0
    except ZeroDivisionError:
        log.error("calcul_echoue", exc_info=True, operation="division")

except ImportError:
    print("structlog non installé")

---

## 8. Synthèse

| Concept | Outil |
|---|---|
| Logger structuré | `structlog.get_logger()` |
| Processeurs | Chaîne de transformation des événements |
| Binding | `log.bind(key=val)` — contexte persistant |
| Contextvars | `structlog.contextvars` — contexte par requête |
| Sortie JSON | `structlog.processors.JSONRenderer()` |
| Sortie dev | `structlog.dev.ConsoleRenderer()` |
| Intégration stdlib | `structlog.stdlib.LoggerFactory()` |

**Règles à retenir :**
- Les logs structurés sont **interrogeables** ; les logs textuels ne le sont pas.
- `structlog` est un wrapper léger ; il peut utiliser `logging` en backend.
- Bindez le contexte (request_id, user_id) au début de chaque requête.
- JSON en production, console colorée en dev.
- Nommez les événements en snake_case, pas en phrases.

---

## 9. Exercices

### Exercice 1 — Premier logger structuré *(facile)*

Configurer `structlog` avec un timestamp ISO et la sortie console. Logger les événements :
- `app_start` avec la version ;
- `config_loaded` avec le nombre de clés ;
- `app_ready` avec le port.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Structlog", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import structlog

structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.ConsoleRenderer(),
    ],
)

log = structlog.get_logger()
log.info("app_start", version="2.1.0")
log.info("config_loaded", nb_cles=15)
log.info("app_ready", port=8080)
```

</details>

### Exercice 2 — Processeur personnalisé *(moyen)*

Écrire un processeur `filtrer_champs_sensibles` qui remplace par `"***"` la valeur de tout champ dont le nom contient `password`, `secret`, `token`, ou `key`. Tester avec `log.info("auth", password="abc", api_key="xyz")`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Structlog", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import structlog

MOTS_SENSIBLES = {"password", "secret", "token", "key"}

def filtrer_champs_sensibles(logger, method_name, event_dict):
    for cle in list(event_dict.keys()):
        if any(mot in cle.lower() for mot in MOTS_SENSIBLES):
            event_dict[cle] = "***"
    return event_dict

structlog.configure(
    processors=[
        filtrer_champs_sensibles,
        structlog.stdlib.add_log_level,
        structlog.dev.ConsoleRenderer(),
    ],
)

log = structlog.get_logger()
log.info("auth_attempt", username="alice", password="abc123", api_key="sk_live_xyz")
```

</details>

### Exercice 3 — Middleware de logging *(moyen)*

Écrire un décorateur `@log_appel` qui logue automatiquement :
- l'entrée dans la fonction (nom, arguments) ;
- la sortie (résultat ou exception) ;
- la durée d'exécution en ms.

Utiliser `structlog` avec binding du nom de fonction.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Structlog", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import structlog
import functools
import time

structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.ConsoleRenderer(),
    ],
)

def log_appel(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        log = structlog.get_logger().bind(fonction=fn.__name__)
        log.info("appel_debut", args=str(args), kwargs=str(kwargs))
        t0 = time.perf_counter()
        try:
            result = fn(*args, **kwargs)
            duree = (time.perf_counter() - t0) * 1000
            log.info("appel_fin", duree_ms=round(duree, 2), resultat=str(result)[:100])
            return result
        except Exception as e:
            duree = (time.perf_counter() - t0) * 1000
            log.error("appel_erreur", duree_ms=round(duree, 2), exception=str(e), exc_info=True)
            raise
    return wrapper

@log_appel
def diviser(a, b):
    return a / b

diviser(10, 3)
try:
    diviser(10, 0)
except ZeroDivisionError:
    pass
```

</details>

### Exercice 4 — Pipeline de log complet *(difficile)*

Implémenter une configuration `structlog` complète pour une application web :
1. Contextvars pour le request_id (généré avec `secrets.token_hex(8)`) ;
2. Processeur qui ajoute le hostname ;
3. Processeur qui filtre les secrets ;
4. JSON en production, console en dev (basé sur `APP_ENV`) ;
5. Un middleware `log_request` qui simule le logging d'une requête HTTP.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Structlog", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import structlog
import secrets
import socket
import os
import time

MOTS_SENSIBLES = {"password", "secret", "token", "authorization"}

def ajouter_hostname(logger, method_name, event_dict):
    event_dict["hostname"] = socket.gethostname()
    return event_dict

def masquer_secrets(logger, method_name, event_dict):
    for cle in list(event_dict.keys()):
        if any(mot in cle.lower() for mot in MOTS_SENSIBLES):
            event_dict[cle] = "***"
    return event_dict

env = os.environ.get("APP_ENV", "dev")

processeurs = [
    structlog.contextvars.merge_contextvars,
    ajouter_hostname,
    masquer_secrets,
    structlog.stdlib.add_log_level,
    structlog.processors.TimeStamper(fmt="iso"),
    structlog.processors.format_exc_info,
]

if env == "production":
    processeurs.append(structlog.processors.JSONRenderer())
else:
    processeurs.append(structlog.dev.ConsoleRenderer())

structlog.configure(processors=processeurs)

def log_request(method, path, handler_fn):
    structlog.contextvars.clear_contextvars()
    request_id = secrets.token_hex(8)
    structlog.contextvars.bind_contextvars(request_id=request_id)

    log = structlog.get_logger()
    log.info("request_start", method=method, path=path)
    t0 = time.perf_counter()

    try:
        status = handler_fn()
        duree = (time.perf_counter() - t0) * 1000
        log.info("request_end", status=status, duree_ms=round(duree, 2))
    except Exception as e:
        duree = (time.perf_counter() - t0) * 1000
        log.error("request_error", status=500, duree_ms=round(duree, 2), exc_info=True)

# Simulation
def handle_login():
    log = structlog.get_logger()
    log.info("auth_check", user="alice", password="secret123")
    return 200

log_request("POST", "/api/login", handle_login)
```

</details>

---

## 10. Ressources

- [`structlog` — documentation officielle](https://www.structlog.org/)
- [The Twelve-Factor App — Logs](https://12factor.net/logs)
- [Structured Logging — best practices](https://www.structlog.org/en/stable/why.html)
- [Module `logging` — documentation Python](https://docs.python.org/3/library/logging.html)